# Stereo template matching/block matching


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from skimage import data
import os

from ultralytics import YOLO
from collections import deque
import glob

plt.ion()

Start by loading a set of corresponding left and right image. We also resize the images since the computations we expect to do can take a long time to process. Resizing is optional, but recommended atleast until you have verified your computer is capable of doing the computations within reasonable time.

In [ ]:
sequence = '34759_final_project_rect/'
calib_file_path = os.path.join(sequence, 'calib_cam_to_cam.txt')

left_cam_dir = os.path.join(sequence, 'seq_02/image_02/data/')
left_cam = sorted(glob.glob(os.path.join(left_cam_dir, '*')))

right_cam_dir = os.path.join(sequence, 'seq_02/image_03/data/')
right_cam = sorted(glob.glob(os.path.join(right_cam_dir, '*')))

In [ ]:
assert len(left_cam) == len(right_cam)

In [ ]:
left_image = cv2.imread(left_cam[50])
right_image = cv2.imread(right_cam[50])

left_image = cv2.cvtColor(left_image, cv2.COLOR_BGR2RGB)
right_image = cv2.cvtColor(right_image, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].imshow(left_image)
axes[0].set_title("Left camera")
axes[0].axis('off')

axes[1].imshow(right_image)
axes[1].set_title("Right camera")
axes[1].axis('off')
plt.show()

In [ ]:
with open(calib_file_path, 'r') as f:
    calibration = {}
    for line in f.readlines():
        try:
            key, values_str = line.split(':')
            key = key.strip()
            values = np.array([float(x) for x in values_str.split()])

            if key in ['P_rect_02', 'P_rect_03']:
                calibration[key] = values.reshape((3, 4))

            elif key in ['S_rect_02', 'S_rect_03']:
                calibration[key] = values.astype(int)
            else:
                calibration[key] = values
        except ValueError:
            continue

In [ ]:
img_left_path = left_cam[50]
img_right_path = right_cam[50]

img_left = cv2.imread(img_left_path, cv2.IMREAD_GRAYSCALE)
img_right = cv2.imread(img_right_path, cv2.IMREAD_GRAYSCALE)

In [ ]:
def plot_disparity(disp):
    plt.figure(figsize=(18, 18))
    plt.imshow(disp, cmap='gray')
    plt.title("Disparity")
    plt.show()

In [ ]:
min_disp = 7
num_disp = 3 * 16
block_size = 5

P1 = 8 * block_size ** 2
P2 = 32 * block_size ** 2

In [ ]:
stereo = cv2.StereoBM_create(numDisparities=num_disp, blockSize=block_size)
stereo.setMinDisparity(min_disp)
stereo.setDisp12MaxDiff(200)
stereo.setUniquenessRatio(10)
stereo.setSpeckleRange(32)
stereo.setSpeckleWindowSize(80)

# Compute disparity
disp = stereo.compute(img_left, img_right).astype(np.float32) / 16.0

plot_disparity(disp)

In [ ]:
stereo = cv2.StereoSGBM_create(
    minDisparity=min_disp,
    numDisparities=num_disp,
    blockSize=block_size,
    P1=P1,
    P2=P2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=32,
    mode=cv2.STEREO_SGBM_MODE_SGBM
)
disp = stereo.compute(img_left, img_right).astype(np.float32) / 16.0

plt.figure(figsize=(18, 18))
plt.imshow(disp, cmap='gray')
plt.title("Disparity")
plt.show()

In [ ]:
ply_header = '''ply
format ascii 1.0
element vertex %(vert_num)d
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
end_header
'''


def write_ply(fn, verts, colors):
    verts = verts.reshape(-1, 3)
    colors = colors.reshape(-1, 3)
    verts = np.hstack([verts, colors])
    with open(fn, 'wb') as f:
        f.write((ply_header % dict(vert_num=len(verts))).encode('utf-8'))
        np.savetxt(f, verts, fmt='%f %f %f %d %d %d ')

In [ ]:
P_rect_02 = calibration['P_rect_02']
P_rect_03 = calibration['P_rect_03']

# Focal length (fx)
f = P_rect_02[0, 0]
fx = P_rect_02[0, 0]
fy = P_rect_02[1, 1]

# Principal point x (cx)
cx = P_rect_02[0, 2]
cx_cam = P_rect_02[0, 2]

# Principal point y (cy)
cy = P_rect_02[1, 2]
cy_cam = P_rect_02[1, 2]


# Baseline term (Tx_rect)
Tx_rect = P_rect_03[0, 3]
Tx = -Tx_rect / fx # 379.7842 / 707.0493 = 0.537 m (This matches T_01/T_03's magnitude)
Q = np.float32([
    [1, 0, 0, -cx_cam],
    [0, 1, 0, -cy_cam],
    [0, 0, 0, fx],
    [0, 0, 1 / Tx, 0] # Last term can be 0 if P[0,3] is 0 and P'[0,3] is f*Tx
])

In [ ]:
points_3D = cv2.reprojectImageTo3D(disp, Q)

img_left_color = cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB)

mask = disp > disp.min()
out_points = points_3D[mask]
out_colors = img_left_color[mask]

write_ply('out.ply', out_points, out_colors)

Next step is to create a disparity map based on the two stereo images. This is done using the same procedure as we did with exercise 1, except now we look for blocks of pixel from the left image in the right image, or vice versa.

## Exercise 2.1
You will have to play with the below parameters until you find a good solution that gives a representative 3D projection. Reflect/search on what those parameters actually mean and how they influence the outcome.

## Exercise 2.2
Once you have a disparity map without too much noise, you can use the below to export you disparity map to a .ply file that you can load and display in a 3D visualization software (e.g. [Meshlab](https://www.meshlab.net/#description)).

**Hint:** To open meshlab type `meshlab` in a terminal. 
To vizualize the point cloud, drag the generated `out.ply` into the meshlab scene or click `File` > `Import Mesh` and navigate to where our source-code is located and open `out.ply`. 

# Kalman Filter functions

## Exercise 2.3

Create a disparity map of the imageset `storage` and display it in meshlab.

# Kalman filter

In [ ]:
class KalmanCV3D:
    """
    Constant-velocity Kalman filter in 3D:
    state = [cx, cy, z, vx, vy, vz]^T
    cx, cy in pixels, z in meters.
    """
    def __init__(self, cx0, cy0, z0=0.0, dt=1.0):
        # State
        self.x = np.array([
            [cx0],
            [cy0],
            [z0],
            [0.0],
            [0.0],
            [0.0]
        ])

        self.P = np.eye(6) * 1000.0

        # Constant-velocity model
        self.F = np.array([
            [1., 0., 0., dt, 0.,  0. ],
            [0., 1., 0., 0., dt,  0. ],
            [0., 0., 1., 0., 0.,  dt ],
            [0., 0., 0., 1., 0.,  0. ],
            [0., 0., 0., 0., 1.,  0. ],
            [0., 0., 0., 0., 0.,  1. ]
        ])

        # Measurement models:
        # full 3D: [cx, cy, z]
        self.H_xyz = np.array([
            [1., 0., 0., 0., 0., 0.],
            [0., 1., 0., 0., 0., 0.],
            [0., 0., 1., 0., 0., 0.],
        ])

        # only 2D: [cx, cy]
        self.H_xy = np.array([
            [1., 0., 0., 0., 0., 0.],
            [0., 1., 0., 0., 0., 0.],
        ])

        # Process noise 
        q_pos_xy = 1e-3
        q_pos_z  = 2e-3
        q_vel_xy = 1
        q_vel_z  = 1e-3
        self.Q = np.diag([
            q_pos_xy, q_pos_xy, q_pos_z,
            q_vel_xy, q_vel_xy, q_vel_z
        ])

        # Measurement noise
        # assume ~5 px std in cx,cy → 25
        # assume ~1 m std in z → 1
        self.R_xyz = np.diag([25.0, 25.0, 4.0])
        self.R_xy  = np.diag([36.0, 36.0])

        self.I = np.eye(6)


    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x

    def update_xy(self, cx_meas, cy_meas):
        z_vec = np.array([[cx_meas],
                          [cy_meas]])
        H = self.H_xy
        R = self.R_xy

        y = z_vec - (H @ self.x)
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.pinv(S)

        self.x = self.x + K @ y
        self.P = (self.I - K @ H) @ self.P
        return self.x

    def update_xyz(self, cx_meas, cy_meas, z_meas):
        z_vec = np.array([[cx_meas],
                          [cy_meas],
                          [z_meas]])
        H = self.H_xyz
        R = self.R_xyz

        y = z_vec - (H @ self.x)
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.pinv(S)

        self.x = self.x + K @ y
        self.P = (self.I - K @ H) @ self.P
        return self.x


# Helper functions

In [ ]:
def iou_xyxy(b1, b2):
    x1a, y1a, x2a, y2a = b1
    x1b, y1b, x2b, y2b = b2

    inter_x1 = max(x1a, x1b)
    inter_y1 = max(y1a, y1b)
    inter_x2 = min(x2a, x2b)
    inter_y2 = min(y2a, y2b)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, x2a - x1a) * max(0, y2a - y1a)
    area_b = max(0, x2b - x1b) * max(0, y2b - y1b)

    union = area_a + area_b - inter_area
    if union <= 0:
        return 0.0
    return inter_area / union


# Config

In [ ]:
# CONFIG
SEC_PATH  = '34759_final_project_rect/seq_02/'
LEFT_DIR  = os.path.join(SEC_PATH, 'image_02/data')
RIGHT_DIR = os.path.join(SEC_PATH, 'image_03/data')

left_files  = sorted([f for f in os.listdir(LEFT_DIR)  if f.endswith('.png')])
right_files = sorted([f for f in os.listdir(RIGHT_DIR) if f.endswith('.png')])

print("Left frames :", len(left_files))
print("Right frames:", len(right_files))

if not left_files:
    raise RuntimeError(f"No PNG files found in {LEFT_DIR}")
if len(left_files) != len(right_files):
    print("WARNING: left/right frame counts differ, assuming same names anyway.")

# Manual bbox in first frame
if SEC_PATH[-7:-1] == 'seq_01':
    bbox_init_xyxy = [466.194319, 139.161762, 557.194320, 332.842544]
elif SEC_PATH[-7:-1] == 'seq_02':
    # Tracker works well for these two initial bounding boxes. They demonstrate 
    # tracking under the fixed cartoon occlusion in the frame.

    bbox_init_xyxy = [436.728458, 161.218615, 476.395123, 254.699107]
    # bbox_init_xyxy = [209.727761, 154.940942, 237.727761, 220.178333]    # This one also demonstrates tracking when  
                                                                         # when biker obscures view
    
else:
    raise RuntimeError("Unknown sequence, add bbox_init_xyxy for it")

x1_init, y1_init, x2_init, y2_init = map(int, bbox_init_xyxy)

# YOLO
model = YOLO("yolov8n.pt")
PERSON_CLASS_ID = 0


# Setup

In [ ]:
# Get all initial bounding boxes from txt file


In [ ]:
# Init from first frame
from collections import deque

# Read first stereo pair
first_left  = os.path.join(LEFT_DIR,  left_files[0])
first_right = os.path.join(RIGHT_DIR, right_files[0])

frame0   = cv2.imread(first_left)
frame0_r = cv2.imread(first_right)
if frame0 is None or frame0_r is None:
    raise RuntimeError("Could not read first stereo pair")

H, W = frame0.shape[:2]

# YOLO on frame 0 + match to manual bbox
res0   = model(frame0, verbose=False)[0]
boxes0 = res0.boxes

x1_0, y1_0, x2_0, y2_0 = x1_init, y1_init, x2_init, y2_init  # fallback = manual
best_iou = 0.0

if boxes0 is not None and len(boxes0) > 0:
    xyxy0 = boxes0.xyxy.cpu().numpy()
    cls0  = boxes0.cls.cpu().numpy()
    for box, c in zip(xyxy0, cls0):
        if int(c) != PERSON_CLASS_ID:
            continue
        x1d, y1d, x2d, y2d = box
        iou = iou_xyxy((x1_init, y1_init, x2_init, y2_init),
                       (x1d, y1d, x2d, y2d))
        if iou > best_iou:
            best_iou = iou
            x1_0, y1_0, x2_0, y2_0 = x1d, y1d, x2d, y2d

if best_iou > 0:
    print(f"Frame 0: matched YOLO person with IoU={best_iou:.3f}")
else:
    print("Frame 0: using manual bbox for init")

w0 = int(x2_0 - x1_0)
h0 = int(y2_0 - y1_0)
cx0 = x1_0 + w0 / 2.0
cy0 = y1_0 + h0 / 2.0

# --- Stereo depth z0 on frame 0 (same ROI logic as main loop) ---
gray0_l = cv2.cvtColor(frame0,   cv2.COLOR_BGR2GRAY)
gray0_r = cv2.cvtColor(frame0_r, cv2.COLOR_BGR2GRAY)
disp0   = stereo.compute(gray0_l, gray0_r).astype(np.float32) / 16.0
pts0    = cv2.reprojectImageTo3D(disp0, Q)

x1 = int(cx0 - w0/2)
x2 = int(cx0 + w0/2)
y1 = int(y1_0 + 0.6*h0)
y2 = int(y1_0 + h0)

x1, x2 = np.clip([x1, x2], 0, W)
y1, y2 = np.clip([y1, y2], 0, H)

z0_meas = None
mask0 = disp0[y1:y2, x1:x2] > 0


if np.any(mask0):
    zvals0 = pts0[y1:y2, x1:x2, 2][mask0]
    zvals0 = zvals0[np.isfinite(zvals0)]
    if zvals0.size > 0:
        z0_meas = float(np.median(zvals0))

if z0_meas is None:
    z0_init = 0.0
    print("WARNING: no stereo depth in frame 0 → z0 = 0")
else:
    z0_init = z0_meas
    print(f"Initial depth z0 = {z0_init:.3f} m")

# --- Kalman + tracking state init ---
kf = KalmanCV3D(cx0, cy0, z0=z0_init, dt=1.0)

last_w, last_h = w0, h0
last_area      = w0 * h0
have_depth     = (z0_meas is not None)

print(f"KF init: cx={cx0:.1f}, cy={cy0:.1f}, z={z0_init:.2f}, w={w0}, h={h0}")

GATE_DIST       = 80.0
MAX_PRED_FRAMES = 60
SPEED_BOOST     = 1.035
SIZE_MIN_RATIO  = 0.6
MAX_Z_JUMP      = 6.0   # m per frame

pred_only_count = 0
z_history       = []
flip            = True

last_box = None
IOU_MIN = 0.3

# Main loop

Note to the loop below. We use the pixel coordinates as kalman updates as these are much less noisy, then we simply map the center coordinates to meters so we have the lable of the bounding box in meters. The z coordinate is predicted based on stereo measurment and kalman updates in meters

In [ ]:
for frame_idx, fname in enumerate(left_files):

    #  Load frames 
    frame   = cv2.imread(os.path.join(LEFT_DIR, fname))
    frame_r = cv2.imread(os.path.join(RIGHT_DIR, fname))
    H, W = frame.shape[:2]

    #  3D KF Predict 
    kf.predict()
    cx_pred, cy_pred, z_pred = kf.x[0,0], kf.x[1,0], kf.x[2,0]

    #  YOLO Detection 
    res = model(frame, verbose=False)[0]
    boxes = res.boxes

    has_det   = False   # YOLO found something consistent with KF | has_det = has detected
    used_meas = False   # we actually updated KF with it          | used_meas = used measurement
    occluded  = False

    person_boxes = []
    if boxes is not None:
        xyxy = boxes.xyxy.cpu().numpy()
        cls  = boxes.cls.cpu().numpy()
        for box, cls_id in zip(xyxy, cls):
            if int(cls_id) == PERSON_CLASS_ID:
                person_boxes.append(box)

    #  Select detection 
    if len(person_boxes) > 0:
        # Pick closest detection to predicted KF center
        best_box = min(
            person_boxes,
            key=lambda b: np.hypot((b[0]+b[2])/2 - cx_pred,
                                   (b[1]+b[3])/2 - cy_pred)
        )

        x1d, y1d, x2d, y2d = best_box
        cx_det = (x1d + x2d) / 2
        cy_det = (y1d + y2d) / 2
        w_meas = max(1, int(x2d - x1d))
        h_meas = max(1, int(y2d - y1d))
        area_meas = w_meas * h_meas

        #  Gating: accept only good matches in terms of Distance from center and IoU
        if (area_meas >= SIZE_MIN_RATIO * last_area) and \
           (np.hypot(cx_det - cx_pred, cy_det - cy_pred) < GATE_DIST):

            used_meas = True
            has_det   = True

            occluded  = False
            pred_only_count = 0

            alpha = 0.2
            size_ratio = area_meas / last_area
            if 0.5 < size_ratio < 2:
                # normal variation  allow update
                last_w = (1-alpha)*last_w + alpha*w_meas
                last_h = (1-alpha)*last_h + alpha*h_meas
                last_area = last_w * last_h
            else:
                # crazy size  do NOT modify last_w/last_h
                used_meas = False  # or treat as occlusion

        else:
            occluded = True
            pred_only_count += 1

    else:
        # No persons at all
        occluded = True
        pred_only_count += 1

    #  BBOX from current KF state  
    cx, cy, z_hat = kf.x[0,0], kf.x[1,0], kf.x[2,0]

    x_draw_c = int(cx - last_w/2)
    y_draw_c = int(cy - last_h/2)

    #  Depth Measurement 
    z_meas = None

    if used_meas:
        gray_l = cv2.cvtColor(frame,   cv2.COLOR_BGR2GRAY)
        gray_r = cv2.cvtColor(frame_r, cv2.COLOR_BGR2GRAY)
        disp   = stereo.compute(gray_l, gray_r).astype(np.float32) / 16.0
        pts    = cv2.reprojectImageTo3D(disp, Q)

        # ROI at feet
        x1, x2 = x_draw_c, x_draw_c + last_w
        y1, y2 = int(y_draw_c + 0.6*last_h), y_draw_c + last_h

        x1, x2 = np.clip([x1, x2], 0, W)
        y1, y2 = np.clip([y1, y2], 0, H)

        x1, x2, y1, y2 = map(int, [x1, x2, y1, y2])

        # Extract ROI arrays
        disp_roi = disp[y1:y2, x1:x2]
        z_roi    = pts[y1:y2, x1:x2, 2]

        mask = (disp_roi > 0) & np.isfinite(z_roi)

        if np.any(mask):                          
            mask &= np.abs(z_roi - z_pred) < 5.0  

            if np.any(mask):
                z_candidates = z_roi[mask]

                # We take the median to get z estimate
                z_meas = float(np.median(z_candidates))

    # 3D KF UPDATE 
    if has_det:
        if z_meas is not None:
            dz = z_meas - z_pred
            if abs(dz) <= MAX_Z_JUMP and z_meas is not None:

                kf.update_xyz(cx_det, cy_det, z_meas)
                used_meas   = True
                last_w      = w_meas
                last_h      = h_meas
                last_area   = area_meas
            else:
                occluded = True
                pred_only_count += 1
                
        else:
            # z_meas is None  treat as unreliable / occlusion
            occluded = True
            pred_only_count += 1

    #  Occlusion compensation 
    if occluded:
        kf.x[3,0] *= SPEED_BOOST   
        kf.x[4,0] *= SPEED_BOOST   

    if pred_only_count > MAX_PRED_FRAMES:
        print("Too many prediction frames — aborting")
        break

    z_history.append(z_pred)

    if occluded and flip:
        z_history.append('just got occluded')
        flip = False
    
    if not occluded and not flip:
        z_history.append('just reappeard')
        flip = True

    # Draw bbox 
    cx, cy, z_hat = kf.x[0,0], kf.x[1,0], kf.x[2,0]

    x_meters = (cx - cx_cam) * z_hat / fx
    y_meters = (cy - cy_cam) * z_hat / fy

    x_draw = cx - last_w/2
    y_draw = cy - last_h/2

    x_draw_c = np.clip(x_draw, 0, W-1)
    y_draw_c = np.clip(y_draw, 0, H-1)
    w_draw_c = max(1, min(last_w, W - x_draw_c))
    h_draw_c = max(1, min(last_h, H - y_draw_c))

    x0 = int(x_draw_c)
    y0 = int(y_draw_c)
    x1 = int(x_draw_c + w_draw_c)
    y1 = int(y_draw_c + h_draw_c)

    frame_vis = frame.copy()
    color = (0,255,0) if used_meas else (0,255,255)

    cv2.rectangle(frame_vis, (x0, y0), (x1, y1), color, 2)

    tag = "YOLO+KF" if used_meas else "KF pred"
    cv2.putText(frame_vis, f"{fname} {tag}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # coord_text = f"cx={cx:.1f} cy={cy:.1f} px  Z={z_hat:.2f} m"
    coord_text = f"X={x_meters:.2f} m Y={y_meters:.2f} m  Z={z_hat:.2f} m"
    cv2.putText(frame_vis, coord_text,
                (x0, max(20, y0 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    plt.clf()
    plt.imshow(cv2.cvtColor(frame_vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Frame {frame_idx:04d}")
    plt.axis('off')
    plt.pause(0.01)


# Tracking mulitple objects at once

In [ ]:
next_track_id = 0

class Track:
    def __init__(self, track_id, cx, cy, z0, w, h, dt=1.0):
        self.id = track_id
        self.kf = KalmanCV3D(cx, cy, z0, dt=dt)

        self.w = w
        self.h = h
        self.area = w * h

        self.missed = 0          
        self.hit = 1             
        self.color = (0, 255, 0) 
tracks = []  # list[Track]


# Fuck this fucking shit below here. I spent too much time tying to make it work but the id tracker from yolo was just not working properly and i think the approach above might be better. Im very up for suggestions

In [ ]:
a = 5 + 'k'

In [ ]:
class KalmanCV3D:
    # State: [X, Y, Z, Vx, Vy, Vz]^T, Meas: [X, Y, Z]^T
    def __init__(self, X0, Y0, Z0, dt=1.0):
        self.dt = dt
        self.x = np.array([[X0], [Y0], [Z0], [0.0], [0.0], [0.0]], dtype=np.float32)

        self.F = np.array([
            [1, 0, 0, dt, 0,  0],
            [0, 1, 0, 0,  dt, 0],
            [0, 0, 1, 0,  0,  dt],
            [0, 0, 0, 1,  0,  0],
            [0, 0, 0, 0,  1,  0],
            [0, 0, 0, 0,  0,  1]
        ], dtype=np.float32)

        self.H = np.array([
            [1, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0],
        ], dtype=np.float32)

        self.P = np.eye(6, dtype=np.float32) * 1
        self.Q = np.eye(6, dtype=np.float32) * 1
        self.R = np.eye(3, dtype=np.float32) * 1

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x.copy()

    def update(self, z):
        z = np.asarray(z, dtype=np.float32).reshape(3, 1)
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        I = np.eye(self.P.shape[0], dtype=np.float32)
        self.P = (I - K @ self.H) @ self.P
        return self.x.copy()


def compute_disparity_and_points3d(left_img, right_img, stereo_matcher, Q):
    grayL = cv2.cvtColor(left_img, cv2.COLOR_BGR2GRAY)
    grayR = cv2.cvtColor(right_img, cv2.COLOR_BGR2GRAY)

    disp = stereo_matcher.compute(grayL, grayR).astype(np.float32) / 16.0
    points_3d = cv2.reprojectImageTo3D(disp, Q)
    return disp, points_3d


def get_3d_from_bbox_center(bbox, points_3d):
    x1, y1, x2, y2 = map(int, bbox)
    x1, x2 = min(x1, x2), max(x1, x2)
    y1, y2 = min(y1, y2), max(y1, y2)

    cx = int(round((x1 + x2) / 2.0))
    cy = int(round((y1 + y2) / 2.0))

    h_img, w_img = points_3d.shape[:2]
    cx = int(np.clip(cx, 0, w_img - 1))
    cy = int(np.clip(cy, 0, h_img - 1))

    X, Y, Z = points_3d[cy, cx]

    w_box = max(1, x2 - x1)
    h_box = max(1, y2 - y1)

    return X, Y, Z, (cx, cy), (h_box, w_box)


def get_bbox_from_3d(point_3d, point_cloud, box_hw=None):

    # Unpack input
    X, Y, Z = point_3d
    h_img, w_img = point_cloud.shape[:2]

    # Default box size (height, width)
    if box_hw is None:
        box_hw = (100, 50)

    h_box, w_box = box_hw

    # Flatten point cloud and mask invalid points
    pts = point_cloud.reshape(-1, 3)
    valid_mask = np.isfinite(pts).all(axis=1)
    if not np.any(valid_mask):
        return None, (None, None)

    valid_pts = pts[valid_mask]
    # Compute squared distances to avoid unnecessary sqrt
    target = np.array([X, Y, Z], dtype=valid_pts.dtype)
    diffs = valid_pts - target
    d2 = np.sum(diffs * diffs, axis=1)
    min_idx = int(np.argmin(d2))

    # Map flattened index back to image coordinates
    valid_flat_indices = np.flatnonzero(valid_mask)
    flat_idx = int(valid_flat_indices[min_idx])
    cy = flat_idx // w_img
    cx = flat_idx % w_img

    # Build bbox centered on (cx, cy), using provided height/width
    half_w = int(round(w_box / 2.0))
    half_h = int(round(h_box / 2.0))

    x1 = int(np.clip(cx - half_w, 0, w_img - 1))
    x2 = int(np.clip(cx + half_w, 0, w_img - 1))
    y1 = int(np.clip(cy - half_h, 0, h_img - 1))
    y2 = int(np.clip(cy + half_h, 0, h_img - 1))

    # Ensure non-empty bbox (at least one pixel wide/high)
    if x2 <= x1:
        x2 = min(w_img - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(h_img - 1, y1 + 1)

    return [x1, y1, x2, y2], (int(cx), int(cy))
    

def main():
    pred_bbox = None
    last_box = None
    # ---------- PATHS ----------
    # Adjust these to your setup
    SEQ_ROOT = "34759_final_project_rect/seq_02"
    LEFT_DIR = os.path.join(SEQ_ROOT, "image_02/data")
    RIGHT_DIR = os.path.join(SEQ_ROOT, "image_03/data")

    left_imgs = sorted(glob.glob(os.path.join(LEFT_DIR, "*.png")))
    right_imgs = sorted(glob.glob(os.path.join(RIGHT_DIR, "*.png")))

    assert len(left_imgs) == len(right_imgs) and len(left_imgs) > 0, "Left/right count mismatch"

    # ---------- YOLO + TRACKER ----------
    model = YOLO("yolov8n.pt")   
    model = YOLO("yolo11n.pt"  )   
       
    TRACKER_CFG = "botsort.yaml"  
    TRACKER_CFG = "bytetrack.yaml"  
    PERSON_CLASS = 0

    # ---------- 3D KF ----------
    FPS = 10.0        # or whatever your sequence frame rate is supposed to emulate
    dt = 1.0 / FPS

    # first_box = [436.728458, 161.218615, 476.395123, 254.699107]
    # first_box = [276.028463, 156.615028, 307.060850, 230.650286]
    # first_box = [209.727761, 154.940942, 237.727761, 220.178333]
    first_box = [244.404816, 154.153304, 265.071484, 218.579195]
    # first_box = [271.163454, 159.400648, 298.926237, 218.175444]


    target_id = None
    kf_3d = None
    target_bbox = None
    first_run = True
    missing = True

    for idx, (lp, rp) in enumerate(zip(left_imgs, right_imgs)):
        frameL = cv2.imread(lp)
        frameR = cv2.imread(rp)
        if frameL is None or frameR is None:
            print(f"Failed to read {lp} or {rp}")
            continue


        # YOLO + ID tracking on LEFT image 
        results = model.track(
            frameL,
            persist=True,
            tracker=TRACKER_CFG,
            verbose=False
        )
        
        if len(results) > 0:
            r = results[0]
            boxes = r.boxes

            if boxes is not None and boxes.id is not None:
                xyxy = boxes.xyxy.cpu().numpy()
                clss = boxes.cls.cpu().numpy().astype(int)
                ids  = boxes.id.cpu().numpy().astype(int)

                # no target yet -> pick one
                if target_id is None:
                    best_iou = -1
                    best_box = None
                    best_id = None
                    
                    for box, c, tid in zip(xyxy, clss, ids):
                        if c != PERSON_CLASS:
                            continue

                        iou = iou_xyxy(box, first_box)

                        if iou > best_iou:
                            best_iou = iou
                            best_box = box
                            best_id = tid

                    if best_box is not None:
                        target_id = best_id
                        target_bbox = best_box         

                else:
                    if kf_3d is not None and point_cloud is not None and isinstance(last_box, np.ndarray):
                        x_hat = kf_3d.predict()
                        Xs, Ys, Zs = x_hat[0, 0], x_hat[1, 0], x_hat[2, 0]

                        # ensure ints and normalized order
                        x1, y1, x2, y2 = map(int, last_box)
                        x1, x2 = min(x1, x2), max(x1, x2)
                        y1, y2 = min(y1, y2), max(y1, y2)

                        last_w = max(1, x2 - x1)   # width in pixels
                        last_h = max(1, y2 - y1)   # height in pixels

                        pred_bbox, (cx, cy) = get_bbox_from_3d((Xs, Ys, Zs), point_cloud, (last_h, last_w))

                    # already have target_id -> find its box
                    missing = True
                    for box, c, tid in zip(xyxy, clss, ids):
                        if tid == target_id:
                            target_bbox = box
                            missing = False
                            break

                    if missing == True:
                        print('Missing Id in yolo')
                        print(idx)
                        
        # If we have bbox for our target ID -> 3D
        if target_bbox is not None and missing == True:
            disp, point_cloud = compute_disparity_and_points3d(frameL, frameR, stereo, Q)
            X, Y, Z, (cx, cy), (h_box, w_box) = get_3d_from_bbox_center(target_bbox, point_cloud)

            last_box = target_bbox

            if kf_3d is None: # only run once
                kf_3d = KalmanCV3D(X, Y, Z, dt=dt)

            x_hat = kf_3d.update([X, Y, Z])
            Xs, Ys, Zs = x_hat[0, 0], x_hat[1, 0], x_hat[2, 0]

            # Draw
            x1, y1, x2, y2 = target_bbox
            x1i, y1i, x2i, y2i = map(int, [x1, y1, x2, y2])

            cv2.rectangle(frameL, (x1i, y1i), (x2i, y2i), (0, 255, 0), 2)
            cv2.circle(frameL, (cx, cy), 3, (0, 0, 255), -1)

            txt = f"id={target_id} Xs={Xs:.2f} Ys={Ys:.2f} Zs={Zs:.2f} "
            cv2.putText(frameL, txt, (x1i, max(0, y1i - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            
            if pred_bbox is not None:
                x1, y1, x2, y2 = pred_bbox
                x1i, y1i, x2i, y2i = map(int, [x1, y1, x2, y2])

                cv2.rectangle(frameL, (x1i, y1i), (x2i, y2i), (0, 0, 255), 2)
                cv2.circle(frameL, (cx, cy), 3, (255, 0, 0), -1)

                txt = f"id={target_id} Xs={Xs:.2f} Ys={Ys:.2f} Zs={Zs:.2f} "
                cv2.putText(frameL, txt, (x1i, max(0, y1i - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        else:
            # Target ID not visible in this frame
            if pred_bbox is not None:
                x1, y1, x2, y2 = pred_bbox
                x1i, y1i, x2i, y2i = map(int, [x1, y1, x2, y2])

                cv2.rectangle(frameL, (x1i, y1i), (x2i, y2i), (0, 255, 0), 2)
                cv2.circle(frameL, (cx, cy), 3, (0, 0, 255), -1)

                txt = f"id={target_id} Xs={Xs:.2f} Ys={Ys:.2f} Zs={Zs:.2f} "
                cv2.putText(frameL, txt, (x1i, max(0, y1i - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                
                cv2.putText(frameL, f"id={target_id} occluded Zs={Zs:.2f}",
                            (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (0, 255, 255), 2)
                

        cv2.putText(frameL, f"frame {idx}", (20, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

        cv2.imshow("Left with ID tracking + 3D", frameL)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break

    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


# OLD

In [ ]:
z_history